# Football Prediction System - Data Exploration

In [ ]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_league_data, LEAGUE_CONFIG
plt.style.use('seaborn-v0_8-whitegrid')
OUTPUTS_DIR = '../outputs'
os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [ ]:
# Load all leagues
LEAGUES = list(LEAGUE_CONFIG.keys())
data = {}
for lg in LEAGUES:
    df = load_league_data(lg)
    if not df.empty:
        data[lg] = df
print(f"Loaded {len(data)} leagues with data")

In [ ]:
# Summary
rows = []
for lg, df in data.items():
    rows.append({
        'League': LEAGUE_CONFIG[lg]['name'],
        'Country': LEAGUE_CONFIG[lg]['country'],
        'Matches': len(df),
        'Teams': df['HomeTeam'].nunique(),
        'Seasons': df['Season'].nunique() if 'Season' in df.columns else 'N/A'
    })
summary = pd.DataFrame(rows)
summary

In [ ]:
# Market stats
stats = []
for lg, df in data.items():
    if 'FTHG' in df.columns and 'FTAG' in df.columns:
        total = df['FTHG'] + df['FTAG']
        stats.append({
            'League': LEAGUE_CONFIG[lg]['name'],
            'Over25': round((total > 2.5).mean() * 100, 1),
            'BTTS': round(((df['FTHG'] > 0) & (df['FTAG'] > 0)).mean() * 100, 1),
            'HomeWin': round((df['FTR'] == 'H').mean() * 100, 1),
            'Draw': round((df['FTR'] == 'D').mean() * 100, 1),
            'AwayWin': round((df['FTR'] == 'A').mean() * 100, 1),
            'AvgGoals': round(total.mean(), 2)
        })
stats_df = pd.DataFrame(stats)
stats_df

In [ ]:
# Plot market rates
names = stats_df['League'].values
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

x = np.arange(len(names))
width = 0.6

axes[0].bar(x, stats_df['Over25'], width, color='steelblue', edgecolor='black')
axes[0].axhline(y=50, color='red', linestyle='--', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].set_title('Over 2.5 Goals Rate (%)')
axes[0].set_ylabel('%')

axes[1].bar(x, stats_df['BTTS'], width, color='#2ecc71', edgecolor='black')
axes[1].axhline(y=50, color='red', linestyle='--', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=45, ha='right')
axes[1].set_title('BTTS Rate (%)')
axes[1].set_ylabel('%')

axes[2].bar(x, stats_df['HomeWin'], width, color='#2ecc71', edgecolor='black', label='Home')
axes[2].bar(x, stats_df['Draw'], width, bottom=stats_df['HomeWin'], color='#f39c12', edgecolor='black', label='Draw')
axes[2].bar(x, stats_df['AwayWin'], width, bottom=stats_df['HomeWin']+stats_df['Draw'], color='#e74c3c', edgecolor='black', label='Away')
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=45, ha='right')
axes[2].set_title('Match Outcomes (%)')
axes[2].set_ylabel('%')
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, '01_market_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/01_market_analysis.png")

In [ ]:
# Goals distribution per league
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes_flat = axes.flatten()

for idx, (lg, df) in enumerate(data.items()):
    if idx >= len(axes_flat):
        break
    if 'FTHG' in df.columns and 'FTAG' in df.columns:
        total_goals = df['FTHG'] + df['FTAG']
        total_goals.hist(ax=axes_flat[idx], bins=range(0, 10), color='steelblue', edgecolor='black')
        avg = total_goals.mean()
        axes_flat[idx].axvline(x=avg, color='red', linestyle='--', label=f'Avg: {avg:.1f}')
        axes_flat[idx].set_title(LEAGUE_CONFIG[lg]['name'])
        axes_flat[idx].set_xlabel('Total Goals')
        axes_flat[idx].legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Total Goals Distribution by League', y=1.02, fontsize=14)
plt.savefig(os.path.join(OUTPUTS_DIR, '01_goals_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/01_goals_distribution.png")

In [ ]:
# Odds availability
odds_cols = ['AvgH', 'AvgD', 'AvgA', 'Avg>2.5', 'Avg<2.5', 'AvgAHH', 'AvgAHA']
print("Odds Availability by League:")
print("=" * 60)
for lg, df in data.items():
    avail = sum(1 for c in odds_cols if c in df.columns)
    print(f"{LEAGUE_CONFIG[lg]['name']:25s}: {avail}/{len(odds_cols)} odds columns")

In [ ]:
# Save combined data
combined = pd.concat([df for df in data.values()], ignore_index=True)
os.makedirs('../data/combined', exist_ok=True)
combined.to_csv('../data/combined/all_leagues.csv', index=False)
print(f"Saved combined dataset: {len(combined)} matches")